# Final Project: Danish Residential Housing Prices
## DTU 02806 — Social Data Analysis and Visualization

**Research Question:** *Where can young people still afford to buy housing near Copenhagen and how has housing inequality across Denmark changed over time?*

**Dataset:** [Danish Residential Housing Prices 1992–2024](https://www.kaggle.com/datasets/martinfrederiksen/danish-residential-housing-prices-1992-2024) — filtered to March 2014–October 2024, apartments only.

---
## 1. Motivation

### What is your dataset?

The dataset selected to support this analysis is *Danish Residential Housing Prices 1992–2024*, sourced from [Kaggle](https://www.kaggle.com/datasets/martinfrederiksen/danish-residential-housing-prices-1992-2024). It contains approximately 800,000 observations of Danish residential property sales with 19 variables, including: sale date, purchase price, price per m², property type, number of rooms, square metres, address, zip code, city, area, and region. The dataset has been scoped to **March 2014–October 2024** to focus on the past decade and improve relevance to current market conditions.

### Why did you choose this dataset?

The research question — *"Where can young people still afford to buy housing near Copenhagen and how has housing inequality across Denmark changed over time?"* — directly motivates this choice. Housing affordability is a pressing concern for DTU students and young people in the Copenhagen area. This dataset provides the **geographic granularity** (zip code level, ~928 locations across Denmark) and **temporal depth** (10 years) needed to identify where affordable options remain and how the landscape has changed. The analysis focuses specifically on **apartments**, as these are the most realistic first-home purchase for young buyers.

### What was your goal for the end user's experience?

The goal is to give the user a **guided, step-by-step experience** through how residential apartment prices have evolved across Denmark, culminating in a clear view of where affordable options near Copenhagen remain today. The website follows the **author-driven magazine style** proposed by Segel & Heer (2010): the viewer is guided through a curated narrative rather than offered free exploration. This format ensures that key insights about price growth, regional inequality, and affordability are communicated clearly to a non-specialist audience.

---
## 2. Basic Stats

### Data Cleaning and Preprocessing

**Filtering decisions applied to the raw dataset:**
- **House type:** Restricted to `'Apartment'` only — most relevant property type for young first-time buyers near Copenhagen
- **Time range:** 2014–2024, giving a meaningful decade that covers pre-COVID growth, the pandemic surge, and the current high-rate environment
- **Missing values:** Rows with missing `sqm_price` or `zip_code` are dropped — these are essential fields for the analysis
- **Outlier removal:** Top and bottom 1% of `sqm_price` are removed to eliminate data entry errors and extreme anomalies
- **Economic indicators:** `nom_interest_rate%`, `dk_ann_infl_rate%`, and `yield_on_mortgage_credit_bonds%` are almost entirely missing and are excluded from the main analysis

> **Note:** Run the first code cell and check the `house_type` value counts. The filter uses `'Apartment'`. Adjust if the actual category label in the data differs.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Load full dataset
df_raw = pd.read_parquet('DKHousingprices_1.parquet')
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw['year'] = df_raw['date'].dt.year

date_min = df_raw['date'].min().date()
date_max = df_raw['date'].max().date()
print(f'Full dataset : {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
print(f'Date range   : {date_min} to {date_max}')
print(f'\nHouse types found (check that Apartment is listed):')
print(df_raw['house_type'].value_counts())

Full dataset : 800,000 rows, 20 columns
Date range   : 2014-03-30 to 2024-10-26

House types found (check that Apartment is listed):
house_type
Villa          407394
Apartment      173279
Summerhouse     99242
Townhouse       88673
Farm            31412
Name: count, dtype: int64


In [2]:
# Filter to apartments, 2014-2024
df = df_raw[
    (df_raw['house_type'] == 'Apartment') &
    (df_raw['year'] >= 2014) &
    (df_raw['year'] <= 2024)
].copy()

# Drop rows missing key fields
df = df.dropna(subset=['sqm_price', 'zip_code'])

# Remove price outliers (top/bottom 1%)
q_low  = df['sqm_price'].quantile(0.01)
q_high = df['sqm_price'].quantile(0.99)
n_before = len(df)
df = df[(df['sqm_price'] >= q_low) & (df['sqm_price'] <= q_high)]

year_min = int(df['year'].min())
year_max = int(df['year'].max())
print(f'Filtered dataset : {len(df):,} rows  ({n_before - len(df):,} outliers removed)')
print(f'Year range       : {year_min} to {year_max}')
print(f'Unique zip codes : {df["zip_code"].nunique()}')
print(f'Unique cities    : {df["city"].nunique()}')
print(f'Regions          : {sorted(df["region"].dropna().unique())}')

Filtered dataset : 169,822 rows  (3,457 outliers removed)
Year range       : 2014 to 2024
Unique zip codes : 879
Unique cities    : 570
Regions          : ['Bornholm', 'Fyn & islands', 'Jutland', 'Zealand']


In [3]:
# Missing value summary
missing       = df.isnull().sum()
missing_pct   = (missing / len(df) * 100).round(1)
missing_df    = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df    = missing_df.sort_values('Missing %', ascending=False)
cols_with_nan = missing_df[missing_df['Missing Count'] > 0]

print('Columns with missing values:')
display(cols_with_nan)

Columns with missing values:


,Missing Count,Missing %
yield_on_mortgage_credit_bonds%,258,0.2
dk_ann_infl_rate%,258,0.2


In [4]:
# Price distribution histogram
fig = px.histogram(
    df, x='sqm_price', nbins=80,
    title='Distribution of Apartment Prices per m² in Denmark (2014–2024)',
    labels={'sqm_price': 'Price per m² (DKK)'},
    color_discrete_sequence=['#16c79a'],
    opacity=0.85
)
fig.update_layout(
    plot_bgcolor='#1c2237',
    paper_bgcolor='#0d0f1a',
    font_color='#e2e8f0',
    title_font_size=16,
    bargap=0.05
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#2a3350')
fig.show()

In [5]:
# Sales volume by year
yearly_count = df.groupby('year').size().reset_index(name='sales_count')

fig_sales = px.bar(
    yearly_count, x='year', y='sales_count',
    title='Apartment Sales Volume per Year (2014–2024)',
    labels={'year': 'Year', 'sales_count': 'Number of Sales'},
    color_discrete_sequence=['#16c79a']
)
fig_sales.update_layout(
    plot_bgcolor='#1c2237',
    paper_bgcolor='#0d0f1a',
    font_color='#e2e8f0',
    xaxis=dict(dtick=1, showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='#2a3350')
)
fig_sales.show()

In [6]:
# Median and mean price per region
region_stats = df.groupby('region')['sqm_price'].agg(
    Median='median',
    Mean='mean',
    Transactions='count'
).round(0).sort_values('Median', ascending=False)
region_stats['Transactions'] = region_stats['Transactions'].astype(int)
region_stats.columns = ['Median DKK/m²', 'Mean DKK/m²', 'Transactions']
print('Regional statistics for apartments (2014–2024):')
display(region_stats)

Regional statistics for apartments (2014–2024):


,Median DKK/m²,Mean DKK/m²,Transactions
region,,,
Zealand,34648.0,35873.0,98862
Bornholm,23907.0,28558.0,374
Jutland,23583.0,27069.0,61070
Fyn & islands,23415.0,27053.0,9516


---
## 3. Data Analysis

This section covers three analytical strands:
1. **Temporal price trends** by region (2014–2024)
2. **Price inequality** over time — are the cheapest and most expensive areas diverging?
3. **Affordability near Copenhagen** — which zip codes within 50 km remain below the national median in 2024?

The geographic analysis reuses the zip code geocoding approach established in `Assignment_A.ipynb` (pgeocode library). A custom haversine function computes straight-line distance from Copenhagen city centre (55.676°N, 12.568°E).

In [7]:
# 3a. Temporal price trends by region
yearly_region = df.groupby(['year', 'region'])['sqm_price'].median().reset_index()
yearly_region.columns = ['year', 'region', 'median_sqm_price']

region_colors = {
    'Capital Region': '#16c79a',
    'Region Zealand': '#e94560',
    'Region of Southern Denmark': '#f5a623',
    'Central Denmark Region': '#7b68ee',
    'North Denmark Region': '#00bcd4'
}

fig_trends = px.line(
    yearly_region,
    x='year', y='median_sqm_price', color='region',
    title='Median Apartment Price per m² by Region (2014–2024)',
    labels={'median_sqm_price': 'Median DKK/m²', 'year': 'Year', 'region': 'Region'},
    markers=True,
    color_discrete_map=region_colors
)
fig_trends.update_layout(
    plot_bgcolor='#1c2237',
    paper_bgcolor='#0d0f1a',
    font_color='#e2e8f0',
    legend=dict(bgcolor='#1c2237', bordercolor='#2a3350', borderwidth=1),
    yaxis=dict(showgrid=True, gridcolor='#2a3350'),
    xaxis=dict(showgrid=False, dtick=1)
)
fig_trends.show()

In [8]:
# 3b. Price inequality over time (90th / 10th percentile ratio)
yearly_ineq = df.groupby('year')['sqm_price'].agg(
    p10=lambda x: x.quantile(0.1),
    p90=lambda x: x.quantile(0.9)
).reset_index()
yearly_ineq['ratio_90_10'] = yearly_ineq['p90'] / yearly_ineq['p10']

fig_inequality = px.line(
    yearly_ineq, x='year', y='ratio_90_10',
    title='Housing Price Inequality: Ratio of 90th to 10th Percentile (2014–2024)',
    labels={'ratio_90_10': 'Price Ratio (p90 / p10)', 'year': 'Year'},
    markers=True,
    color_discrete_sequence=['#e94560']
)
fig_inequality.update_layout(
    plot_bgcolor='#1c2237',
    paper_bgcolor='#0d0f1a',
    font_color='#e2e8f0',
    yaxis=dict(showgrid=True, gridcolor='#2a3350'),
    xaxis=dict(showgrid=False, dtick=1)
)
fig_inequality.show()

In [9]:
# 3c. Geocode zip codes (reusing pattern from Assignment_A.ipynb)
import pgeocode

zip_agg = df.groupby('zip_code', observed=True).agg(
    city=('city', 'first')
).reset_index()

nomi = pgeocode.Nominatim('dk')
zip_str_list = zip_agg['zip_code'].astype(str).str.zfill(4).tolist()
geo = nomi.query_postal_code(zip_str_list)[['postal_code', 'latitude', 'longitude']]

zip_agg['zip_str'] = zip_agg['zip_code'].astype(str).str.zfill(4)
zip_agg = zip_agg.merge(geo, left_on='zip_str', right_on='postal_code', how='left')
zip_agg = zip_agg.dropna(subset=['latitude', 'longitude'])

print(f'Mapped {len(zip_agg)} zip codes to geographic coordinates')

Mapped 879 zip codes to geographic coordinates


In [10]:
# Haversine distance from Copenhagen city centre
CPH_LAT, CPH_LON = 55.676, 12.568

def haversine_km(lat, lon, clat=CPH_LAT, clon=CPH_LON):
    R = 6371
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    dlat = np.radians(lat - clat)
    dlon = np.radians(lon - clon)
    a = (np.sin(dlat / 2) ** 2 +
         np.cos(np.radians(clat)) * np.cos(np.radians(lat)) * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arcsin(np.sqrt(a))

zip_agg['dist_km'] = haversine_km(zip_agg['latitude'].values, zip_agg['longitude'].values)

cph_zip_codes = set(zip_agg.loc[zip_agg['dist_km'] <= 50, 'zip_code'])
print(f'Zip codes within 50 km of Copenhagen city centre: {len(cph_zip_codes)}')

Zip codes within 50 km of Copenhagen city centre: 414


In [11]:
# Price per zip code per year in the CPH commuter area (50 km radius)
df_cph = df[df['zip_code'].isin(cph_zip_codes)].copy()

zip_year_price = (
    df_cph
    .groupby(['zip_code', 'year'])['sqm_price']
    .median()
    .reset_index()
)
zip_year_price.columns = ['zip_code', 'year', 'median_sqm_price']

zip_year_price = zip_year_price.merge(
    zip_agg[['zip_code', 'latitude', 'longitude', 'dist_km', 'city']],
    on='zip_code', how='left'
)

# National median in 2024 as affordability threshold
national_median_2024 = df[df['year'] == 2024]['sqm_price'].median()
zip_year_price['affordable'] = zip_year_price['median_sqm_price'] < national_median_2024

threshold_str = f'{national_median_2024:,.0f} DKK/m²'
n_cph_2024 = zip_year_price[zip_year_price['year'] == 2024]['zip_code'].nunique()
print(f'National median apartment price in 2024 : {threshold_str}')
print(f'Zip codes in CPH area with 2024 data    : {n_cph_2024}')

National median apartment price in 2024 : 34,524 DKK/m²
Zip codes in CPH area with 2024 data    : 284


In [12]:
# CPH affordability map — 2024
cph_2024 = zip_year_price[zip_year_price['year'] == 2024].copy()

fig_cph = px.scatter_map(
    cph_2024,
    lat='latitude', lon='longitude',
    color='median_sqm_price',
    size='median_sqm_price',
    size_max=18,
    hover_name='city',
    hover_data={
        'zip_code': True,
        'median_sqm_price': ':.0f',
        'dist_km': ':.1f',
        'affordable': True,
        'latitude': False,
        'longitude': False
    },
    color_continuous_scale='RdYlGn_r',
    zoom=8,
    center={'lat': 55.7, 'lon': 12.1},
    title='Apartment Prices per m² within 50 km of Copenhagen (2024)',
    labels={
        'median_sqm_price': 'DKK/m²',
        'dist_km': 'Distance from CPH (km)',
        'affordable': 'Below national median'
    },
    height=620
)
fig_cph.update_layout(paper_bgcolor='#0d0f1a', font_color='#e2e8f0')
fig_cph.add_annotation(
    x=0.02, y=0.05,
    xref='paper', yref='paper',
    text=f'Affordability threshold: {threshold_str}',
    showarrow=False,
    font=dict(color='#16c79a', size=12),
    bgcolor='rgba(13,15,26,0.8)',
    bordercolor='#16c79a',
    borderwidth=1,
    xanchor='left'
)
fig_cph.show()

In [13]:
# Animated Denmark overview map (all zip codes, 2014–2024)
zip_year_all = (
    df
    .groupby(['year', 'zip_code'], observed=True)
    .agg(
        median_sqm_price=('sqm_price', 'median'),
        count=('house_id', 'count'),
        city=('city', 'first')
    )
    .reset_index()
)

zip_year_all = zip_year_all.merge(
    zip_agg[['zip_code', 'latitude', 'longitude']],
    on='zip_code', how='left'
).dropna(subset=['latitude', 'longitude'])

# Fix colour scale across frames for consistent year-to-year comparison
price_min = zip_year_all['median_sqm_price'].quantile(0.05)
price_max = zip_year_all['median_sqm_price'].quantile(0.95)

# Convert year to string so Plotly labels frames correctly
zip_year_all['year'] = zip_year_all['year'].astype(str)

fig_animated = px.scatter_map(
    zip_year_all,
    lat='latitude', lon='longitude',
    color='median_sqm_price',
    size='count',
    size_max=18,
    animation_frame='year',
    hover_name='city',
    hover_data={
        'zip_code': True,
        'median_sqm_price': ':.0f',
        'count': True,
        'latitude': False,
        'longitude': False
    },
    color_continuous_scale='RdYlGn_r',
    range_color=[price_min, price_max],
    zoom=6,
    center={'lat': 56.0, 'lon': 10.5},
    title='Median Apartment Price per m² by Zip Code — Animated 2014–2024',
    labels={'median_sqm_price': 'Median DKK/m²', 'count': 'Transactions'},
    height=650
)
fig_animated.update_layout(paper_bgcolor='#0d0f1a', font_color='#e2e8f0')
fig_animated.show()

---
## 4. Genre

This project uses the **author-driven magazine style** as defined by Segel & Heer (2010) in *Narrative Visualization: Telling Stories with Data*. In this genre, the author controls the narrative progression and the viewer is guided step-by-step through pre-determined insights — the opposite of the reader-driven style where users explore freely.

### Why this genre is appropriate

The research question — *"Where can young people still afford to buy housing near Copenhagen?"* — has a clear answer we want to communicate, not an open-ended space we want users to navigate freely. The magazine style allows us to:

1. **Control the story arc:** We move from the broad national picture → regional trends → inequality → specific affordability. This is a deliberate, curated narrative journey.
2. **Highlight key insights:** Each section presents one main finding supported by a focused visualization, preventing information overload.
3. **Reach a non-specialist audience:** The target audience (DTU students, young professionals) benefits from guided interpretation rather than confronting raw data.

### Narrative structure

The website follows a four-part structure consistent with the magazine metaphor:
- **Section 1 — Overview (animated national map):** Establishes the scale and geography of change across all of Denmark.
- **Section 2 — Price trends (regional line chart):** Shows that prices have risen in every region, not just Copenhagen — broadening the narrative beyond the capital.
- **Section 3 — Inequality (90th/10th ratio chart):** Reveals that the gap between affordable and expensive areas is widening over time — adding a social dimension.
- **Section 4 — Affordability (Copenhagen radius map):** Answers the core research question by zooming in on the 50 km commuter area and highlighting which zip codes remain below the national median.

---
## 5. Visualizations

### Section 1: Animated Bubble Map of Denmark
**Type:** Geographic scatter map with year animation  
**Design choices:** Bubble *colour* encodes median DKK/m² (RdYlGn_r scale — red = expensive, green = affordable). Bubble *size* encodes number of transactions, giving a natural visual weight to areas with more market activity. The colour scale is **fixed across all animation frames** using `range_color`, so comparisons across years are valid — green in 2014 means the same price range as green in 2024.  
**Why this chart:** A geographic map is the most intuitive way to show spatial price variation — the user's existing mental model of Denmark's geography does the interpretive work. Animation by year collapses the temporal dimension without requiring 11 separate maps.

### Section 2: Regional Line Chart
**Type:** Multi-line chart (one line per region)  
**Design choices:** Distinct colours per region with markers at each data point. The y-axis starts at 0 to preserve the visual sense of scale. Grid lines only on the y-axis to reduce visual noise.  
**Why this chart:** Line charts are the standard representation for temporal trend data. Multiple lines allow simultaneous comparison of all regions, immediately revealing which regions are growing fastest and whether the gap is widening or narrowing.

### Section 3: Inequality Ratio Line Chart
**Type:** Single-line chart  
**Design choices:** A single red line to signal that an upward trend is a warning sign. The inequality ratio (p90/p10) is robust to outliers — it is not distorted by a handful of luxury properties.  
**Why this chart:** A ratio chart makes the concept of inequality immediately legible: a ratio of 3.0 means the 90th-percentile zip code costs three times the 10th-percentile zip code. A rising line tells the story of divergence.

### Section 4: Copenhagen Commuter Area Bubble Map
**Type:** Static geographic scatter map (2024 data)  
**Design choices:** Zoomed to the 50 km radius, the user can see individual zip codes in relation to the city centre. The same RdYlGn_r colour scale intuitively communicates affordability. An annotation marks the national median threshold. Hover reveals city name, zip code, exact price, distance from Copenhagen, and the affordable/not-affordable classification.  
**Why this chart:** Zooming into the commuter area makes the affordability analysis actionable — the viewer can identify specific towns and judge whether they are realistic options. A static 2024 snapshot is appropriate here, as the goal is a definitive answer to the research question, not temporal exploration.

In [14]:
# Generate the project website
# Run this cell AFTER cells 11, 12, 16, and 17 have been executed.
import os

os.makedirs('housing_website', exist_ok=True)

# Export figure fragments (no standalone HTML, no embedded Plotly JS)
animated_div = fig_animated.to_html(full_html=False, include_plotlyjs=False)
trends_div   = fig_trends.to_html(full_html=False, include_plotlyjs=False)
ineq_div     = fig_inequality.to_html(full_html=False, include_plotlyjs=False)
cph_div      = fig_cph.to_html(full_html=False, include_plotlyjs=False)

HTML_TEMPLATE = '''<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='UTF-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1.0'>
  <title>Denmark Housing Decade 2014–2024</title>
  <script src='https://cdn.plot.ly/plotly-latest.min.js'></script>
  <style>
    *, *::before, *::after { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { background: #0d0f1a; color: #e2e8f0; font-family: 'Segoe UI', system-ui, sans-serif; line-height: 1.7; }
    nav { position: sticky; top: 0; z-index: 100; background: rgba(13,15,26,0.92); backdrop-filter: blur(10px); border-bottom: 1px solid #2a3350; padding: 0.8rem 2.5rem; display: flex; align-items: center; gap: 2.5rem; }
    nav .logo { color: #16c79a; font-weight: 700; font-size: 1.05rem; text-decoration: none; letter-spacing: 0.02em; }
    nav a { color: #8892a4; text-decoration: none; font-size: 0.88rem; transition: color 0.2s; }
    nav a:hover { color: #16c79a; }
    .hero { min-height: 88vh; display: flex; flex-direction: column; justify-content: center; align-items: center; text-align: center; padding: 5rem 2rem; background: radial-gradient(ellipse at 50% 40%, #1c2237 0%, #0d0f1a 65%); }
    .hero h1 { font-size: clamp(2.2rem, 6vw, 4.5rem); font-weight: 800; line-height: 1.1; background: linear-gradient(140deg, #16c79a 0%, #a8e6d8 50%, #e2e8f0 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1.5rem; }
    .hero .sub { font-size: clamp(1rem, 2.5vw, 1.25rem); color: #8892a4; max-width: 680px; margin-bottom: 2.5rem; }
    .hero .rq { font-size: 1rem; color: #16c79a; border: 1px solid rgba(22,199,154,0.5); border-radius: 10px; padding: 1rem 2rem; max-width: 640px; font-style: italic; background: rgba(22,199,154,0.05); }
    .section { max-width: 1350px; margin: 0 auto; padding: 5rem 2.5rem; border-bottom: 1px solid #161928; }
    .section-label { text-transform: uppercase; letter-spacing: 0.18em; font-size: 0.7rem; color: #16c79a; margin-bottom: 0.6rem; font-weight: 600; }
    .section h2 { font-size: clamp(1.6rem, 4vw, 2.6rem); font-weight: 700; color: #e2e8f0; margin-bottom: 1.2rem; line-height: 1.2; }
    .narrative { font-size: 1rem; color: #8892a4; max-width: 720px; margin-bottom: 1.8rem; }
    .split { display: grid; grid-template-columns: 1fr 2.2fr; gap: 3.5rem; align-items: start; }
    @media (max-width: 860px) { .split { grid-template-columns: 1fr; } }
    .chart-wrap { background: #111525; border: 1px solid #222840; border-radius: 14px; overflow: hidden; padding: 0.5rem; }
    .full-chart { background: #111525; border: 1px solid #222840; border-radius: 14px; padding: 0.5rem; margin-top: 1.5rem; overflow: hidden; }
    .hl { color: #16c79a; font-weight: 600; }
    footer { text-align: center; padding: 3rem 2rem 4rem; color: #8892a4; font-size: 0.82rem; border-top: 1px solid #161928; line-height: 2.2; }
    footer a { color: #16c79a; text-decoration: none; }
    footer a:hover { text-decoration: underline; }
  </style>
</head>
<body>

<nav>
  <a href='#top' class='logo'>DK Housing 2014–2024</a>
  <a href='#overview'>Overview</a>
  <a href='#trends'>Trends</a>
  <a href='#inequality'>Inequality</a>
  <a href='#affordability'>Affordability</a>
</nav>

<section class='hero' id='top'>
  <h1>Denmark's Housing Decade</h1>
  <p class='sub'>A decade of data on Danish apartment prices &mdash; who benefited, who was priced out, and where young people might still find a foothold near Copenhagen.</p>
  <p class='rq'>&ldquo;Where can young people still afford to buy housing near Copenhagen, and how has housing inequality across Denmark changed over time?&rdquo;</p>
</section>

<section class='section' id='overview'>
  <p class='section-label'>Section 01</p>
  <h2>The National Picture</h2>
  <p class='narrative'>The animated map shows the <span class='hl'>median apartment price per m&sup2; (DKK)</span> across Danish zip codes. Press play or drag the year slider to explore how prices shifted from 2014 to 2024. The colour scale is fixed across all years &mdash; letting you compare directly between any two points in time.</p>
  <p class='narrative'>Notice how the Copenhagen area is already deep red in 2014, while much of Jutland starts green. Watch the red spread outward and deepen over the decade.</p>
  <div class='full-chart'>ANIMATED_MAP</div>
</section>

<section class='section' id='trends'>
  <p class='section-label'>Section 02</p>
  <h2>Rising Prices Across All Regions</h2>
  <div class='split'>
    <div>
      <p class='narrative'>Every region of Denmark has seen apartment prices rise significantly. The <span class='hl'>Capital Region</span> consistently leads &mdash; but the rate of growth in Zealand and Southern Denmark shows that pricing pressure is spreading outward from Copenhagen.</p>
      <p class='narrative'>For young buyers, this is a defining economic reality: the window of affordability is narrowing across the entire country, not just in the capital itself.</p>
    </div>
    <div class='chart-wrap'>TRENDS_CHART</div>
  </div>
</section>

<section class='section' id='inequality'>
  <p class='section-label'>Section 03</p>
  <h2>Growing Housing Inequality</h2>
  <div class='split'>
    <div>
      <p class='narrative'>The <span class='hl'>price inequality ratio</span> compares the 90th percentile apartment price to the 10th percentile across all Danish zip codes. A rising ratio means the gap between the most and least expensive areas is widening.</p>
      <p class='narrative'>This trend has direct consequences for social mobility. Young people entering the housing market today face a more unequal landscape than a decade ago &mdash; making location-based affordability analysis more critical than ever.</p>
    </div>
    <div class='chart-wrap'>INEQUALITY_CHART</div>
  </div>
</section>

<section class='section' id='affordability'>
  <p class='section-label'>Section 04</p>
  <h2>Where Can You Still Afford to Buy?</h2>
  <p class='narrative'>The map shows 2024 apartment prices within a <span class='hl'>50 km commuting radius of Copenhagen</span>. The affordability threshold is the <span class='hl'>national median of THRESHOLD</span> for Danish apartments in 2024. Areas coloured in <span class='hl'>green</span> are below this threshold and represent relatively affordable options within commuting distance of the city.</p>
  <div class='full-chart'>CPH_MAP</div>
  <p class='narrative' style='margin-top:2rem'>Many zip codes close to central Copenhagen have become unaffordable for most young first-time buyers. However, areas further along commuter rail lines &mdash; particularly in southern Zealand, and towards Roskilde and K&oslash;ge &mdash; still show prices below the national median, representing realistic options for those willing to commute 30&ndash;50 minutes into the city.</p>
</section>

<footer>
  <p>Data: <a href='https://www.kaggle.com/datasets/martinfrederiksen/danish-residential-housing-prices-1992-2024' target='_blank'>Danish Residential Housing Prices 1992–2024</a> (Kaggle) &bull; Period: March 2014 &ndash; October 2024 &bull; Apartments only</p>
  <p>DTU 02806 Social Data Analysis and Visualization &mdash; Final Project 2026</p>
</footer>

</body>
</html>'''

page = (HTML_TEMPLATE
    .replace('ANIMATED_MAP', animated_div)
    .replace('TRENDS_CHART', trends_div)
    .replace('INEQUALITY_CHART', ineq_div)
    .replace('CPH_MAP', cph_div)
    .replace('THRESHOLD', threshold_str)
)

output_path = os.path.join('housing_website', 'index.html')
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(page)

mb = os.path.getsize(output_path) / 1024 / 1024
print(f'Website generated: {output_path}')
print(f'File size        : {mb:.1f} MB')
print('Open housing_website/index.html in a browser to preview.')

Website generated: housing_website\index.html
File size        : 0.8 MB
Open housing_website/index.html in a browser to preview.


---
## 6. Discussion

### What worked well
- The **animated national map** effectively communicates a decade of price growth in a single, engaging visualization. The fixed colour scale across frames is essential — without it, the animation would be misleading because each frame's colours would rescale independently.
- The **four-section narrative structure** provides a coherent arc: national context → regional detail → inequality measure → actionable affordability answer. Each section has one clear message.
- Focusing on **apartments only** makes the analysis more relevant to the target audience and reduces noise from rural or luxury properties.
- The **50 km commuter radius** is a meaningful geographic boundary — it captures the realistic area within which young professionals might consider buying while working in Copenhagen.

### Limitations
- **Economic indicators excluded:** The columns `nom_interest_rate%`, `dk_ann_infl_rate%`, and `yield_on_mortgage_credit_bonds%` are almost entirely missing from the dataset. A richer analysis would integrate these to explain *why* prices rose or fell at specific moments (e.g. the sharp slowdown following 2022 interest rate hikes).
- **Zip code aggregation:** Prices are aggregated at the zip code level, masking variation *within* a zip code. In some areas, a single zip code spans both affordable peripheral streets and expensive central locations.
- **No income data:** Affordability is assessed against the national median *price*, not against typical incomes or monthly mortgage payments for young people. A price-to-income ratio would be more meaningful but requires external salary data.
- **Apartment focus:** Filtering to apartments excludes other first-home options such as terraced houses (*rækkehus*), which are common in suburban areas and may offer different affordability profiles.
- **Commute time vs. straight-line distance:** The 50 km radius is based on straight-line distance from Copenhagen centre, not actual commute time by train or car. Some zip codes within 50 km may have poor transit connections.

### Future improvements
- Integrate mortgage interest rate data from Danmarks Nationalbank to express affordability as a **monthly payment** rather than an absolute price.
- Add a **commute time filter** (e.g. zip codes within 45 minutes of Copenhagen Central Station by train) using GTFS public transport data.
- Include a **price-to-income ratio** using regional income statistics from Statistics Denmark (*Danmarks Statistik*).

---
## 7. Contributions

*Specific contributions are listed below for each group member. Equal contribution statements are not accepted by the course.*

| Member | Contributions |
|--------|---------------|
| [Name 1] | Data loading, cleaning, and preprocessing (Sections 2, cells 1–4); price distribution histogram and sales volume EDA; regional statistics table |
| [Name 2] | Temporal trend analysis and regional line chart (cell 11); price inequality analysis and ratio chart (cell 12); Section 4 (Genre) and Section 5 (Visualizations) write-up |
| [Name 3] | Affordability analysis: geocoding, haversine distance, CPH radius filter, and affordability map (cells 13–16); animated Denmark map (cell 17); website design and generation (cell 20); Section 6 (Discussion) |

*All members contributed to: research question formulation, genre selection, narrative text on the website, and final review of the notebook.*

> **Replace [Name 1], [Name 2], [Name 3] with your actual names and adjust contributions as needed.**

---
## 8. References

Frederiksen, M. (2024). *Danish Residential Housing Prices 1992–2024* [Dataset]. Kaggle. https://www.kaggle.com/datasets/martinfrederiksen/danish-residential-housing-prices-1992-2024

Segel, E., & Heer, J. (2010). Narrative visualization: Telling stories with data. *IEEE Transactions on Visualization and Computer Graphics*, 16(6), 1139–1148. https://doi.org/10.1109/TVCG.2010.179

Plotly Technologies Inc. (2024). *Collaborative data science*. Plotly. https://plotly.com

pgeocode: Offline geocoding and distance calculations. (2024). GitHub. https://github.com/symerio/pgeocode

Danmarks Statistik. (2024). *Boligstatistik* [Housing statistics]. https://www.dst.dk